<div dir="rtl">

# مشروع الترم ٣: مساعد ذكي بيفهم الأوامر

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdelrahman-abozarifa04/ai-for-kids-course/blob/main/notebooks/teens/term-3/12-assistant-project.ipynb)

في الترم ١ عملنا شات بوت **بالقواعد** واتكسر بسرعة. دلوقتي هنعمل مساعد **بيتعلم من أمثلة** يفهم نية المستخدم (intent classification)، حتى لو كتبها بطريقة عمره ما شافها.

**المكونات:**
1. 🧠 موديل يصنّف الرسالة لنية (وقت، طقس، نكتة، حساب، تذكير...).
2. 🔧 "أدوات" تنفّذ كل نية.
3. 🎙️ (اختياري في Colab) صوت: تحويل الكلام لنص والعكس.

</div>

In [ ]:
import re
import random
from datetime import datetime, timedelta

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline, make_union
from sklearn.model_selection import cross_val_score

<div dir="rtl">

## ١. بيانات التدريب: أمثلة لكل نية

</div>

In [ ]:
TRAINING = {
    "time": ["what time is it", "tell me the time", "current time please", "what's the time now",
             "do you know the time", "time?", "how late is it", "is it late", "clock please"],
    "date": ["what is the date today", "what day is it", "today's date", "which day is today",
             "tell me the date", "what's today", "what month is it", "is today friday", "date please"],
    "joke": ["tell me a joke", "make me laugh", "say something funny", "i want a joke",
             "do you know any jokes", "cheer me up", "i'm sad", "another joke", "be funny"],
    "math": ["what is 5 plus 3", "calculate 12 times 4", "how much is 100 divided by 5",
             "what's 7 minus 2", "compute 3 * 9", "solve 15 + 27", "add 8 and 2",
             "multiply 6 by 7", "what is 2 + 2"],
    "reminder": ["remind me to study in 10 minutes", "set a reminder to drink water",
                 "remind me to call mom", "don't let me forget the homework",
                 "reminder: football at 5", "remind me in 30 minutes to stretch",
                 "remember to buy bread", "alert me to pray", "remind me about the exam"],
    "greeting": ["hello", "hi there", "hey assistant", "good morning", "salam", "ahlan",
                 "hey", "hi", "good evening"],
    "goodbye": ["bye", "goodbye", "see you later", "that's all thanks", "good night", "exit",
                "see ya", "i'm done", "talk to you later"],
}
texts = [t for intent, examples in TRAINING.items() for t in examples]
labels = [intent for intent, examples in TRAINING.items() for _ in examples]
print(len(texts), "examples,", len(TRAINING), "intents")

<div dir="rtl">

## ٢. ندرّب الـ intent classifier

بنستخدم نوعين features مع بعض (`make_union`): **كلمات كاملة** و**قطع حروف** (علشان الأخطاء الإملائية والعامية).

</div>

In [ ]:
intent_model = make_pipeline(
    make_union(TfidfVectorizer(), TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4))),
    LogisticRegression(max_iter=1000, C=10),
)
print(f"Cross-validated accuracy: {cross_val_score(intent_model, texts, labels, cv=3).mean():.0%}")
intent_model.fit(texts, labels)

# جمل الموديل ما شافهاش قبل كده
for msg in ["yo what's the time", "pls remind me to sleep", "can you add 4 and 6", "i'm bored make me smile"]:
    probs = intent_model.predict_proba([msg])[0]
    best = probs.argmax()
    print(f"{msg!r:<35} -> {intent_model.classes_[best]} ({probs[best]:.0%})")

<div dir="rtl">

## ٣. الأدوات: كل نية ليها دالة

</div>

In [ ]:
REMINDERS: list[tuple[datetime, str]] = []
JOKES = ["Why do programmers prefer dark mode? Because light attracts bugs. 🐛",
         "I'd tell you a UDP joke, but you might not get it.",
         "There are 10 types of people: those who understand binary and those who don't."]


def do_math(msg: str) -> str:
    words = {"plus": "+", "add": "+", "and": "+", "minus": "-", "times": "*", "multiplied by": "*",
             "divided by": "/", "x": "*"}
    expr = msg.lower()
    for w, op in words.items():
        expr = re.sub(rf"\b{w}\b", op, expr)
    expr = "".join(ch for ch in expr if ch in "0123456789+-*/. ()")
    nums = re.findall(r"\d+\.?\d*", expr)
    if not nums or not re.search(r"[+\-*/]", expr):
        return "I couldn't find a calculation in that."
    try:
        # آمن: الحروف المسموحة أرقام وعمليات بس
        result = eval(expr, {"__builtins__": {}}, {})
    except (SyntaxError, ZeroDivisionError):
        return "That calculation doesn't work."
    return f"{expr.strip()} = {round(result, 4)}"


def do_reminder(msg: str, now: datetime | None = None) -> str:
    now = now or datetime.now()
    minutes = re.search(r"(\d+)\s*min", msg)
    when = now + timedelta(minutes=int(minutes.group(1))) if minutes else now + timedelta(hours=1)
    task = re.sub(r"(remind me|set a reminder|reminder:?|don't let me forget|to|in \d+\s*minutes?)", " ", msg)
    task = re.sub(r"\s+", " ", task).strip() or "something"
    REMINDERS.append((when, task))
    return f"⏰ OK! I'll remind you to '{task}' at {when:%H:%M}."


TOOLS = {
    "time": lambda m: f"It's {datetime.now():%H:%M}.",
    "date": lambda m: f"Today is {datetime.now():%A %d %B %Y}.",
    "joke": lambda m: random.choice(JOKES),
    "math": do_math,
    "reminder": do_reminder,
    "greeting": lambda m: "Hi! Ask me the time, a joke, some math, or to remind you of something.",
    "goodbye": lambda m: "Bye! 👋",
}

<div dir="rtl">

## ٤. المساعد كامل + "مش متأكد"

لو الموديل مش واثق (احتمال أقل من حد معين)، الأحسن يسأل بدل ما يهبد. ده مبدأ مهم في الذكاء الاصطناعي المسؤول.

</div>

In [ ]:
CONFIDENCE = 0.35

def assistant(msg: str) -> str:
    probs = intent_model.predict_proba([msg])[0]
    best = probs.argmax()
    if probs[best] < CONFIDENCE:
        return "🤔 I'm not sure what you mean. Can you say it another way?"
    return TOOLS[intent_model.classes_[best]](msg)

for msg in ["hey", "what's 15 times 3", "remind me to study in 20 minutes", "tell me something funny",
            "which day is today", "what is the capital of mars bananas", "see ya"]:
    print(f"You: {msg}\nAssistant: {assistant(msg)}\n")

<div dir="rtl">

## ٥. 🎙️ صوت (في Colab بس)

```python
!pip install -q gTTS SpeechRecognition
from gtts import gTTS
from IPython.display import Audio

reply = assistant("tell me a joke")
gTTS(reply, lang="en").save("reply.mp3")
Audio("reply.mp3", autoplay=True)
```

وللتعرف على الكلام: سجّل ملف صوتي، وحوّله لنص بـ `speech_recognition`، وابعت النص لـ `assistant()`. **الفكرة:** الصوت مجرد طبقة فوق نفس المخ.

</div>

<div dir="rtl">

## 🏋️ مهام المشروع

1. **نية جديدة:** ضيف `weather` بـ ٦ أمثلة على الأقل، وخلي الرد يقول إنه محتاج إنترنت (أو استخدم API لو عايز تتحدى نفسك).
2. **عربي:** ضيف أمثلة عامية لكل نية ("الساعة كام"، "فكرني أذاكر").
3. **قياس:** اكتب ١٠ جمل اختبار ما استخدمتهاش في التدريب، واحسب دقة المساعد عليها.

<details><summary>حل مهمة ٢</summary>

```python
arabic = {
    "time": ["الساعة كام", "قولي الساعة", "الساعة كام دلوقتي"],
    "date": ["النهارده كام", "النهارده ايه", "تاريخ النهارده"],
    "joke": ["قولي نكتة", "ضحكني", "عايز اضحك"],
    "reminder": ["فكرني اذاكر", "فكرني اشرب مية", "متخلنيش انسى الواجب"],
    "greeting": ["ازيك", "صباح الخير", "اهلا"],
    "goodbye": ["سلام", "باي", "تصبح على خير"],
}
for intent, examples in arabic.items():
    texts += examples
    labels += [intent] * len(examples)
intent_model.fit(texts, labels)
for msg in ["الساعة كام يا معلم", "فكرني انام", "ضحكني شوية"]:
    print(msg, "->", intent_model.predict([msg])[0])
```
</details>

</div>

In [ ]:
# اكتب الحل هنا

<div dir="rtl">

## ✅ معايير التقييم
| المعيار | الدرجة |
|---|---|
| المساعد بيفهم ٧ نوايا على الأقل من جمل جديدة | ٣٠٪ |
| ضاف نية جديدة + أداة شغالة | ٢٠٪ |
| قاس الدقة على جمل اختبار منفصلة | ٢٠٪ |
| التعامل مع عدم الثقة ("مش متأكد") | ١٠٪ |
| عرض حي (live demo) | ٢٠٪ |

</div>